In [ ]:
import numpy as np
import pandas as pd
import math
import os
import argparse


In [ ]:
sc = 'column30'  # maximum possible solar collector potential 
heat = 'column3'  # heating demand

min_sc_area = 10 # min installed solar collector area
sc_module_area  = 2.5 # solar collector module area
min_sc_num = math.floor(min_sc_area / sc_module_area) # min number of solar collector module for installation

In [ ]:
def cost(area):
    dic = {'cost_nos': 0,
           'ins_cost_nos': 0,
           'capex_nos': 0,
           'opex_nos': 0}
    
    if area != 0:
        dic['cost_nos'] = 300 * area + 800
        dic['ins_cost_nos'] = 0.25 * dic['cost_nos']
        dic['capex_nos'] = dic['cost_nos'] + dic['ins_cost_nos']
        dic['opex_nos'] = 2.5 * area + 100
        
    return dic
    


def gas(h): 
    h = h / 0.8
    if h <= 2000:
        return 6.00 * 12 + (20.47 + 2.226) * h / 100 
    elif h <= 10000:
        return 8.00 * 12 + (18.62 + 2.226) * h / 100
    elif h <= 30000:
        return 12.00 * 12 + (17.73 + 2.226)* h / 100
    elif h <= 100000:
        return 14.00 * 12 + (17.61 + 2.226) * h / 100
    elif h <= 300000:
        return 24.00 * 12 + (17.37 + 2.226) * h / 100
    else:
        return 60.00 * 12 + (17.2 + 2.226) * h / 100
    
def initial_constants(dic, area):    
    dic['gas_savings_nos'] = dic['gas_cost_nos']  - dic['remain_gas_cost_nos']
    result = cost(area)
    dic['capex_nos'] = result['capex_nos']
    dic['opex_nos'] = result['opex_nos']

    return dic


def npv_cal(dic, i):
    gas_increase = 0.04492035398 # average gas increase rate from euro stat 
    interest_rate = 0.0337 # interest rate from bundesbank
    c = dic['gas_savings_nos'] * (1 + gas_increase) ** i - dic['opex_nos']
    cur = c / (1 + interest_rate)**i 
    return cur

def npv(dic, years):

    dic['n_nos_0'] = -1 * dic['capex_nos']
    dic['npv_nos'] = dic['n_nos_0']
    for i in range(1, years+1):
        dic['n_nos_'+str(i)] = npv_cal(dic, i)
        dic['npv_nos'] = dic['npv_nos'] + dic['n_nos_'+str(i)]

    return dic

def cal_sc(o_df, scale):
    df = o_df.copy()  # Make a copy of the dataframe
    
    scaled_sc = 'scaled_sc_nos'  # solar collector potential if installing num of solar collector + storage system
    o = 'over_supply_nos'  # solar collector potential over supply
    us = 'under_supply_nos'  # solar collector potential used for heating demand at the same time stamp
    need = 'need_nos'  # remaining heating demand after using solar potential at the same time stamp
    
    df[scaled_sc] = df[sc] * scale
    df[o] = np.where(df[scaled_sc] <= df[heat], 0,  df[scaled_sc] - df[heat])
    df[us] = np.where(df[o] == 0, df[scaled_sc], df[heat])
    df[need] = df[heat] - df[us]

        
    return df, df[need].sum(), df[heat].sum() - df[need].sum()

def install_systems(df, max_area):    
    max_num = int(max_area // sc_module_area)
    original_num = max_area / sc_module_area
    
    best_dic = {'heat': df[heat].sum()}
    best_dic['gas_cost_nos'] = gas(best_dic['heat'])
    best_dic['need_nos'] = best_dic['heat']
    best_dic['remain_gas_cost_nos'] = best_dic['gas_cost_nos']
    best_dic['num_sc_nos'] = 0
    best_dic['total_installed_area_nos'] = 0
    best_dic['saved_nos'] = 0

    best_dic = initial_constants(best_dic, 0)
    best_dic = npv(best_dic, 25)
    
    return_df = df.copy()
    
    if max_num < min_sc_num:
        return best_dic, df
    
    
    for i in range(max_num, min_sc_num-1, -1):
        total_area = i * sc_module_area
        
        temp_df, need, saved = cal_sc(df, i/original_num)
        
        temp_dic = best_dic.copy()
        temp_dic['need_nos'] = need
        temp_dic['remain_gas_cost_nos'] =  gas(temp_dic['need_nos'])
        temp_dic['num_sc_nos'] = i
        temp_dic['total_installed_area_nos'] = total_area
        temp_dic = initial_constants(temp_dic, total_area)
        temp_dic['saved_nos'] = saved
        
        temp_dic = npv(temp_dic, 25)

        if temp_dic['npv_nos'] >= best_dic['npv_nos']:
            best_dic = temp_dic.copy()
            return_df = temp_df.copy()

            
    return best_dic, return_df

def main(inp, outp):
    overview_df = pd.read_csv(inp)
    
    prefix = "/no_storage/"
    
    summary = {'iri': [],
               'num_sc_nos': [],
               'total_installed_area_nos': [],
               'need_nos': [],
               'saved_nos': [],
               'capex_nos': [],
               'opex_nos': [],
               'npv_nos': []}
    
    for j in range(0, 26):
        summary['n_nos_'+str(j)] = []
    
    c = 0
    overview_df = overview_df[(overview_df['ps'] == 'y') | (overview_df['postcode'].notna())]
    overview_df = overview_df[overview_df['roof_area']>=min_sc_area]
    
    for i, row in overview_df.iterrows(): 
        
        if os.path.exists(prefix + row['tableName'] + ".csv"):
            summary['iri'].append(row['iri'])
    
            df = pd.read_csv(prefix + row['tableName'] + ".csv")
            
            if row['roof_area'] >= min_sc_area:
                result, df_result = install_systems(df, row['roof_area'])
                
                summary['num_sc_nos'].append(result['num_sc_nos'])
                summary['total_installed_area_nos'].append(result['total_installed_area_nos'])
                summary['capex_nos'].append(result['capex_nos'])
                summary['opex_nos'].append(result['opex_nos'])
                summary['need_nos'].append(result['need_nos'])
                summary['saved_nos'].append(result['saved_nos'])
                
                for j in range(0, 26):
                    summary['n_nos_'+str(j)].append(result['n_nos_'+str(j)])   
                summary['npv_nos'].append(result['npv_nos'])
                df_result.to_csv(prefix + row['tableName'] + ".csv", index=False)
                if result['npv_nos'] > 0:
                    c += 1
                    print(row['roof_area'], ', npv,', result['npv_nos'], ',', c)
            else:
                summary['num_sc_nos'].append(0)
                summary['total_installed_area_nos'].append(0)
                summary['capex_nos'].append(0)
                summary['need_nos'].append(0)
                summary['need_nos'].append(0)
                summary['saved_nos'].append(0)
                
                for j in range(0, 26):
                    summary['n_nos_'+str(j)].append(0)   
                summary['npv_nos'].append(0)
            
        print(i)  
        
    
    battery_df = pd.DataFrame.from_dict(summary)    
    battery_df.to_csv(outp, index=False)


In [ ]:

if __name__ == '__main__':
    parser = argparse.ArgumentParser()

    # add arguments to the parser
    parser.add_argument("inp") # time series csv file name
    parser.add_argument("outp") # npv result csv file name

    # parse the arguments
    args = parser.parse_args()
    main(args.inp, args.outp)